&nbsp;

&nbsp;

&nbsp;

# Antisense Transcript Servey

- この研究では、overlap している transcript で、かつ独自の cis-regulatory element を有する transcript を探索する

In [1]:
import datetime
import os
import pathlib

import polars as pl
import polars.selectors as cs
from IPython.display import display

pl.Config.set_fmt_str_lengths(200)
os.environ["POLARS_MAX_THREADS"] = "16"

&nbsp;

&nbsp;

&nbsp;

## 0. Preparation

- 以下のコマンドで、GENCODE release 49 の GTF file を BED 形式に変換
- `cwltool --outdir ./Data/Database/GENCODE_v49/ ./Tools/conver2bed.cwl` 
- ここから、exon レベルとかではなく、transcript level で重なっているかどうかを調査するため、filtering する
- 念のため、GTFファイルでも実行できるようにしておいた

In [2]:
# gencode_v49_gtf = pl.read_csv(
#     "../../Data/Database/GENCODE_v49/gencode.v49.chr_patch_hapl_scaff.annotation.gtf",
#     separator="\t",
#     has_header=False,
#     n_threads=16,
#     comment_prefix="#"
# )

# gencode_v49_gtf_only_transcript = (
#     gencode_v49_gtf
#     .filter(pl.col("column_3") == "transcript")
# )

# # gencode_v49_gtf_only_transcript.write_csv(
# #     "../../Data/Database/GENCODE_v49/gencode.v49.chr_patch_hapl_scaff.annotation_transcript_id_only.gtf",
# #     separator="\t",
# #     include_header=False
# # )

# pl.Config.set_tbl_rows(2)
# display(gencode_v49_gtf_only_transcript)

In [3]:
gencode_v49_bed = pl.read_csv(
    "../../Data/Database/GENCODE_v49/gencode.v49.annotation_transcript_id.bed",
    separator="\t",
    has_header=False,
    n_threads=16
)

gencode_v49_bed_only_transcript = (
    gencode_v49_bed
    .filter(pl.col("column_8") == "transcript")
)

# gencode_v49_bed_only_transcript.write_csv(
#     "../../Data/Database/GENCODE_v49/gencode.v49.annotation_transcript_id_only.bed",
#     separator="\t",
#     include_header=False
# )

pl.Config.set_tbl_rows(2)
display(gencode_v49_bed_only_transcript)

column_1,column_2,column_3,column_4,column_5,column_6,column_7,column_8,column_9,column_10
str,i64,i64,str,str,str,str,str,str,str
"""GL000008.2""",83331,200303,"""ENST00000741402.1""",""".""","""+""","""HAVANA""","""transcript""",""".""","""gene_id ""ENSG00000296732.1""; transcript_id ""ENST00000741402.1""; gene_type ""lncRNA""; gene_name ""ENSG00000296732""; transcript_type ""lncRNA""; transcript_name ""ENST00000741402""; level 2; tag ""Ensembl_cano…"
…,…,…,…,…,…,…,…,…,…
"""chrY""",57213905,57214730,"""ENST00000972827.1""",""".""","""-""","""HAVANA""","""transcript""",""".""","""gene_id ""ENSG00000310542.1""; transcript_id ""ENST00000972827.1""; gene_type ""lncRNA""; gene_name ""ENSG00000310542""; transcript_type ""lncRNA""; transcript_name ""ENST00000972827""; level 2; tag ""TAGENE"";"""


&nbsp;

&nbsp;

&nbsp;

&nbsp;

## 1. `bedtools intersect`

- `-S, -wo` をつけた以下のCWLファイルで実行
- (bed file) `cwltool --outdir ./out/transcript_classification/antisense_transcript/ ./Tools/bedtools_intersect_antisense.cwl`
- (GTF file) `cwltool --outdir ./out/transcript_classification/antisense_transcript/ ./Tools/bedtools_intersect_antisense_gtf.cwl`
- NOTE: ここで作成した `num_match` カラムは、exon とかの重なりではなく、transcript model 全体で重なっているかどうかの base count であることに注意！

In [4]:
bedfile_intersect_antisense = pl.read_csv(
    "../../out/transcript_classification/antisense_transcript/gencode.v49.annotation_transcript_id_only_opposite_strand_intersecting_gencode.v49.annotation_transcript_id_only.bed",
    separator="\t",
    has_header=False,
    n_threads=16,
    new_columns=[
        "chr_a",
        "start_a",
        "end_a",
        "transcript_id_a",
        "score_a",
        "strand_a",
        "annotation_a",
        "attribute_a",
        "itemRgb_a",
        "metadata_a",
        "chr_b",
        "start_b",
        "end_b",
        "transcript_id_b",
        "score_b",
        "strand_b",
        "annotation_b",
        "attribute_b",
        "itemRgb_b",
        "metadata_b",
        "num_match"
    ]
).drop(
    [
        "score_a",
        "score_b",
        "annotation_a",
        "attribute_a",
        "itemRgb_a",
        "annotation_b",
        "attribute_b",
        "itemRgb_b",
        "metadata_a",
        "metadata_b"
    ]
).with_columns(
    # Overlap as a fraction of each transcript span (introns included).
    (
        pl.col("num_match") / (pl.col("end_a") - pl.col("start_a"))
    ).alias("frac_match_a"),
    (
        pl.col("num_match") / (pl.col("end_b") - pl.col("start_b"))
    ).alias("frac_match_b")
)

display(bedfile_intersect_antisense)

chr_a,start_a,end_a,transcript_id_a,strand_a,chr_b,start_b,end_b,transcript_id_b,strand_b,num_match,frac_match_a,frac_match_b
str,i64,i64,str,str,str,i64,i64,str,str,i64,f64,f64
"""GL000008.2""",83331,200303,"""ENST00000741402.1""","""+""","""GL000008.2""",197399,208592,"""ENST00000741649.1""","""-""",2904,0.024826,0.259448
…,…,…,…,…,…,…,…,…,…,…,…,…
"""chrY""",57212183,57215474,"""ENST00000972824.1""","""-""","""chrY""",57209886,57212186,"""ENST00000972775.1""","""+""",3,0.000912,0.001304


In [5]:
# # GTF intersect, projected onto the same columns as bedfile_intersect_antisense.
# # GTF start is 1-based; subtract 1 so it matches the 0-based BED start.
# gtf_intersect_antisense = (
#     pl.read_csv(
#         "../../out/transcript_classification/antisense_transcript/gencode.v49.chr_patch_hapl_scaff.annotation_transcript_id_only_opposite_strand_intersecting_gencode.v49.chr_patch_hapl_scaff.annotation_transcript_id_only.tsv",
#         separator="\t",
#         has_header=False,
#         n_threads=16,
#         new_columns=[
#             "chr_a",
#             "source_a",
#             "feature_a",
#             "start_a",
#             "end_a",
#             "score_a",
#             "strand_a",
#             "frame_a",
#             "attributes_a",
#             "chr_b",
#             "source_b",
#             "feature_b",
#             "start_b",
#             "end_b",
#             "score_b",
#             "strand_b",
#             "frame_b",
#             "attributes_b",
#             "num_match",
#         ],
#     )
#     .with_columns(
#         (pl.col("start_a") - 1).alias("start_a"),
#         (pl.col("start_b") - 1).alias("start_b"),
#         pl.col("attributes_a").str.extract(r'transcript_id "([^"]+)"', 1).alias("transcript_id_a"),
#         pl.col("attributes_b").str.extract(r'transcript_id "([^"]+)"', 1).alias("transcript_id_b"),
#     )
#     .select(
#         "chr_a",
#         "start_a",
#         "end_a",
#         "transcript_id_a",
#         "strand_a",
#         "chr_b",
#         "start_b",
#         "end_b",
#         "transcript_id_b",
#         "strand_b",
#         "num_match",
#     )
# )

# display(gtf_intersect_antisense)

# pair_key = ["transcript_id_a", "transcript_id_b"]
# value_cols = [
#     "chr_a",
#     "start_a",
#     "end_a",
#     "strand_a",
#     "chr_b",
#     "start_b",
#     "end_b",
#     "strand_b",
#     "num_match",
# ]

# print(bedfile_intersect_antisense.height, gtf_intersect_antisense.height)

# # Pairs present on only one side.
# bed_only = bedfile_intersect_antisense.join(gtf_intersect_antisense, on=pair_key, how="anti")
# gtf_only = gtf_intersect_antisense.join(bedfile_intersect_antisense, on=pair_key, how="anti")

# # Same ordered pair, but coordinates or overlap differ.
# mismatched = (
#     bedfile_intersect_antisense.join(
#         gtf_intersect_antisense,
#         on=pair_key,
#         how="inner",
#         suffix="_gtf",
#         validate="1:1",
#     )
#     .filter(
#         (pl.col("chr_a") != pl.col("chr_a_gtf"))
#         | (pl.col("start_a") != pl.col("start_a_gtf"))
#         | (pl.col("end_a") != pl.col("end_a_gtf"))
#         | (pl.col("strand_a") != pl.col("strand_a_gtf"))
#         | (pl.col("chr_b") != pl.col("chr_b_gtf"))
#         | (pl.col("start_b") != pl.col("start_b_gtf"))
#         | (pl.col("end_b") != pl.col("end_b_gtf"))
#         | (pl.col("strand_b") != pl.col("strand_b_gtf"))
#         | (pl.col("num_match") != pl.col("num_match_gtf"))
#     )
# )

# display(bed_only.height, gtf_only.height, mismatched.height)
# display(bed_only.head(5), gtf_only.head(5), mismatched.head(5))

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

## 2. `fanta.bio` CRE Integration Process

- HN-score calculation process を行った transcript (つまり、発現定量できている transcript) + TSS ± 500 bp にCREが紐づいている list と統合
- fanta.bio の CRE が紐づかなかった場合は、今回は考慮しない
- 片方の transcript (suffix: _a) と、もう片方のtranscript (suffix: _b) でそれぞれ作成する
- transcript level で、cre を集約する

&nbsp;

### Filtering by `tag`

- GENCODE release 49 における `tag` 情報でフィルタリング
- 解釈を容易にするために、"readthrough_transcript" などは除外する

In [ ]:
# https://www.gencodegenes.org/pages/tags.html
list_remove_tag = [
    "sequence_error", # transcript contains at least 1 non-canonical splice junction that is associated with a known or novel genome sequence error.
    "readthrough_gene", # protein-coding gene that has a readthrough transcript.
    "readthrough_transcript", # a transcript that overlaps two or more independent loci but is considered to belong to a third, separate locus.
    "mRNA_start_NF", # the mRNA start could not be confirmed.
    "mRNA_end_NF", # the mRNA end could not be confirmed.
    "non_canonical_TEC", # the transcript has a non-canonical splice site that needs experimental confirmation.
    "inferred_transcript_model", # transcript model is not supported by a single piece of transcript evidence. May be supported by multiple fragments of transcript evidence or by combining different evidence sources e.g. protein homology, RNA-seq data, published experimental data.
    "non_submitted_evidence", # a splice variant for which supporting evidence has not been submitted to databases, i.e. the model is based on literature or collaborator evidence.
    "NMD_likely_if_extended", # codon if the transcript were longer but cannot currently be annotated as NMD as does not fulfil all criteria - most commonly lack of an intron downstream of the stop codon.
    "low_sequence_quality", # transcript supported by transcript evidence that, while ampping best-in-genome, shows regions of poor sequence quality.
    "reference_genome_error", # locus overlaps a sequence error or an assembly error in the reference genome that affects its annotation (e.g., 1 or 2bp insertion/deletion, substitution causing premature stop codon). The main effect is that affected transcripts that would have had a CDS are currently annotated without one.
    "retained_intron_CDS", # internal intron of CDS portion of transcript is retained.
    "retained_intron_final", # final intron of CDS portion of transcript is retained.
    "retained_intron_first" # first intron of CDS portion of transcript is retained.
]

In [7]:
HNscore_all_fantabio = pl.read_parquet(
    "../../out/fantabio/parquet_file/HNscore_transcript_all_fantabio_v261001.parquet"
).filter(
    pl.col("cre_id").is_not_null() # fanta.bio の CRE が紐づかなかった場合は、今回は考慮しない
).filter(
    # Drop a row when its tag list shares any element with list_remove_tag
    pl.col("tags").list.set_intersection(list_remove_tag).list.len().fill_null(0) == 0 # NOTE: ここで、list_remove_tag に含まれる tag を持つ row を除外している
)

display(HNscore_all_fantabio)

transcript_id,UP2,DOWN2,UNCHANGED2,HN2,UP2_stranded,DOWN2_stranded,UNCHANGED2_stranded,HN2_stranded,gene_symbol,HGNC_ID,transcript_type,gene_id,tags,strand,number_of_tags,RNACentral_ID,corrrespondence,polyA_features,gene_source,annotation_remarks,annotation_remark_count,cre_id,cre_chrom,cre_chrom_start,cre_chrom_end,cre_name,tss_nearest_transcript_ids,tss_nearest_transcript_distance,transcript_distance,screen_ccres,chip_atlas_antigens,silencerdbs,fantom5_cage_peaks,fantom5_enhancers,directionality,fantabio_cre_class,itemRgb,itemRgb_name
str,i64,i64,i64,i64,i64,i64,i64,i64,str,str,str,str,list[str],str,u32,str,str,list[str],str,list[str],u32,str,str,i64,str,str,list[str],str,i64,list[struct[2]],list[struct[3]],list[struct[5]],list[struct[2]],list[struct[1]],f64,str,str,str
"""ENST00000309758.6""",352,17,37,335,232,4,15,228,"""HSPA6""","""HGNC:5239""","""protein_coding""","""ENSG00000173110.8""","[""basic"", ""Ensembl_canonical"", … ""CCDS""]","""+""",6,null,null,null,"""ensembl_havana_gene_homo_sapiens""","[""MANE_select""]",1,"""FCHS_28105""","""chr1""",161523745,"""161524487""","""cp1@HSPA6""","[""GQ891451""]","""0""",53,"[{""EH38E2843853"",""pELS""}, {""EH38E2843855"",""PLS""}, {""EH38E3981533"",""pELS""}]","[{""STAG2"",4103,[{""SRX9975614"",4103}, {""SRX9975613"",3500}, {""SRX26159208"",562}]}, {""HSF1"",4038,[{""SRX16646095"",4038}, {""SRX16646111"",3572}, … {""SRX13845684"",554}]}, … {""GABPA"",500,[{""SRX2635792"",500}]}]","[{""HS_00013400"",""H1 BMP4 Derived Mesendoderm Cultured Cells"",""Mesendoderm cell"",""Others"",""SVM""}, {""HS_00013401"",""H1 Cells"",""Embryonic stem cell"",""Stem cell"",""SVM""}, … {""HS_00013403"",""Foreskin Fibroblast Primary Cells skin02"",""Prepuce"",""Skin"",""SVM""}]",null,null,0.982111,"""PLA""","""204,0,0""","""red"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""ENST00000549545.1""",11,265,130,-254,8,174,69,-166,"""COQ10A""","""HGNC:26515""","""retained_intron""","""ENSG00000135469.15""",[],"""+""",0,null,null,null,"""ensembl_havana_gene_homo_sapiens""",null,0,"""FCHS_96252""","""chr12""",56268140,"""56268446""","""cp6@COQ10A""","[""ENST00000549545.1""]","""-23""",-23,"[{""EH38E4064794"",""pELS""}]",null,"[{""HS_00173371"",""H1 BMP4 Derived Mesendoderm Cultured Cells"",""Mesendoderm cell"",""Others"",""SVM""}, {""HS_02053667"",""Thyroid Gland"",""Thyroid gland"",""Others"",""gkmSVM""}]",null,null,-0.368421,"""ELA""","""255,202,74""","""yellow"""


In [8]:
# join for suffix: _a
HNscore_all_fantabio_a = HNscore_all_fantabio.select(
    "transcript_id",
    "gene_id",
    "gene_symbol",
    "transcript_type",
    "HN2",
    "HN2_stranded",
    "cre_id",
    "fantabio_cre_class",
).group_by(
    "transcript_id",
    "gene_id",
    "gene_symbol",
    "transcript_type",
    "HN2",
    "HN2_stranded"
).agg(
    pl.struct(
        [
            pl.col("cre_id"),
            pl.col("fantabio_cre_class"),
        ]
    ).alias("cre_info_a")
).rename(
    {
        "transcript_id": "transcript_id_a",
        "gene_id": "gene_id_a",
        "gene_symbol": "gene_symbol_a",
        "transcript_type": "transcript_type_a",
        "HN2": "HN2_a",
        "HN2_stranded": "HN2_stranded_a",
    }
).with_columns(
    # Number of fanta.bio CREs linked to this transcript.
    pl.col("cre_info_a").list.len().alias("n_cre_a")
)

display(HNscore_all_fantabio_a)

transcript_id_a,gene_id_a,gene_symbol_a,transcript_type_a,HN2_a,HN2_stranded_a,cre_info_a,n_cre_a
str,str,str,str,i64,i64,list[struct[2]],u32
"""ENST00000573966.1""","""ENSG00000167858.13""","""TEKT1""","""retained_intron""",-2,-2,"[{""FCHS_166029"",""PLA""}, {""FCHS_166030"",""ELA""}]",2
…,…,…,…,…,…,…,…
"""ENST00000613823.2""","""ENSG00000169764.17""","""UGP2""","""protein_coding""",-2,-4,"[{""FCHS_224270"",""ELA""}, {""FCHS_224271"",""PLA""}, {""FCHS_224272"",""PLA""}]",3


In [9]:
# join for suffix: _b
HNscore_all_fantabio_b = HNscore_all_fantabio.select(
    "transcript_id",
    "gene_id",
    "gene_symbol",
    "transcript_type",
    "HN2",
    "HN2_stranded",
    "cre_id",
    "fantabio_cre_class",
).group_by(
    "transcript_id",
    "gene_id",
    "gene_symbol",
    "transcript_type",
    "HN2",
    "HN2_stranded"
).agg(
    pl.struct(
        [
            pl.col("cre_id"),
            pl.col("fantabio_cre_class"),
        ]
    ).alias("cre_info_b")
).rename(
    {
        "transcript_id": "transcript_id_b",
        "gene_id": "gene_id_b",
        "gene_symbol": "gene_symbol_b",
        "transcript_type": "transcript_type_b",
        "HN2": "HN2_b",
        "HN2_stranded": "HN2_stranded_b"
    }
).with_columns(
    # Number of fanta.bio CREs linked to this transcript.
    pl.col("cre_info_b").list.len().alias("n_cre_b")
)

display(HNscore_all_fantabio_b)

transcript_id_b,gene_id_b,gene_symbol_b,transcript_type_b,HN2_b,HN2_stranded_b,cre_info_b,n_cre_b
str,str,str,str,i64,i64,list[struct[2]],u32
"""ENST00000736489.1""","""ENSG00000225377.7""","""NRSN2-AS1""","""lncRNA""",26,21,"[{""FCHS_249879"",""ELA""}, {""FCHS_249880"",""PLA""}, {""FCHS_249881"",""ELA""}]",3
…,…,…,…,…,…,…,…
"""ENST00000804872.1""","""ENSG00000237883.5""","""DGUOK-AS1""","""lncRNA""",-5,-1,"[{""FCHS_226316"",""PLA""}, {""FCHS_521357"",""PLA""}, … {""FCHS_226319"",""ELA""}]",4


&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

## 3. JOIN & Filtering Condition

- 共有している CRE があり、overlap が小さければ、おそらく divergent transcript の可能性が考えられるが、このようなペアも残しておく
- ここで得られるペアは、HN-score が算出されている + fantabio CRE と紐づいている transcript で、かつ 重なっていることが最低条件
- filtering condition として以下の定義を追加
- あくまで、本研究における定義であることに注意


1. **共有する CRE のうち、fanta.bio で Promoter-level activity (PLA) が 0 の pair を残す (PLA が共有している場合、divergent transcript の可能性が高い)**
2. 両方の transcript の長さに対して、半分以上 overlap していること

&nbsp;

In [10]:
bedfile_intersect_antisense_join = bedfile_intersect_antisense.join(
    HNscore_all_fantabio_a,
    on=["transcript_id_a"],
    how="inner",
    validate="m:1"
).join(
    HNscore_all_fantabio_b,
    on=["transcript_id_b"],
    how="inner",
    validate="m:1"
).with_columns(
    # Shared CREs as "cre_id|class". set_intersection does not accept structs.
    pl.col("cre_info_a")
    .list.eval(
        pl.element().struct.field("cre_id")
        + "|"
        + pl.element().struct.field("fantabio_cre_class")
    )
    .list.set_intersection(
        pl.col("cre_info_b").list.eval(
            pl.element().struct.field("cre_id")
            + "|"
            + pl.element().struct.field("fantabio_cre_class")
        )
    )
    .alias("shared_cre")
).with_columns(
    pl.col("shared_cre").list.len().fill_null(0).alias("n_shared_cre"),
    # Count the same shared CREs separately for each fanta.bio class.
    pl.col("shared_cre")
    .list.eval(pl.element().filter(pl.element().str.ends_with("|ELA")))
    .list.len()
    .fill_null(0)
    .alias("n_shared_cre_ELA"),
    pl.col("shared_cre")
    .list.eval(pl.element().filter(pl.element().str.ends_with("|PLA")))
    .list.len()
    .fill_null(0)
    .alias("n_shared_cre_PLA"),
).drop(
    "shared_cre"
).filter(
    (pl.col("n_shared_cre_PLA") == 0) # PLA が共有している場合、divergent transcript の可能性が高いため、除外
    & (pl.col("frac_match_a") >= 0.5)
    & (pl.col("frac_match_b") >= 0.5)
    & (pl.col("gene_id_a") != pl.col("gene_id_b")) # 同じ gene に紐づいている場合、除外 (ありえないが、念のため)
)

pl.Config.set_tbl_rows(10)
display(bedfile_intersect_antisense_join)


chr_a,start_a,end_a,transcript_id_a,strand_a,chr_b,start_b,end_b,transcript_id_b,strand_b,num_match,frac_match_a,frac_match_b,gene_id_a,gene_symbol_a,transcript_type_a,HN2_a,HN2_stranded_a,cre_info_a,n_cre_a,gene_id_b,gene_symbol_b,transcript_type_b,HN2_b,HN2_stranded_b,cre_info_b,n_cre_b,n_shared_cre,n_shared_cre_ELA,n_shared_cre_PLA
str,i64,i64,str,str,str,i64,i64,str,str,i64,f64,f64,str,str,str,i64,i64,list[struct[2]],u32,str,str,str,i64,i64,list[struct[2]],u32,u32,u32,u32
"""chr1""",765078,827475,"""ENST00000743835.1""","""-""","""chr1""",778746,810063,"""ENST00000412115.2""","""+""",31317,0.501899,1.0,"""ENSG00000225880.7""","""LINC00115""","""lncRNA""",-6,-1,"[{""FCHS_25"",""ELA""}, {""FCHS_26"",""PLA""}]",2,"""ENSG00000237491.11""","""LINC01409""","""lncRNA""",9,12,"[{""FCHS_19"",""PLA""}, {""FCHS_20"",""ELA""}]",2,0,0,0
"""chr1""",765078,827475,"""ENST00000743835.1""","""-""","""chr1""",778757,810062,"""ENST00000670700.1""","""+""",31305,0.501707,1.0,"""ENSG00000225880.7""","""LINC00115""","""lncRNA""",-6,-1,"[{""FCHS_25"",""ELA""}, {""FCHS_26"",""PLA""}]",2,"""ENSG00000237491.11""","""LINC01409""","""lncRNA""",0,3,"[{""FCHS_19"",""PLA""}, {""FCHS_20"",""ELA""}]",2,0,0,0
"""chr1""",765078,827475,"""ENST00000743835.1""","""-""","""chr1""",778759,810065,"""ENST00000657896.2""","""+""",31306,0.501723,1.0,"""ENSG00000225880.7""","""LINC00115""","""lncRNA""",-6,-1,"[{""FCHS_25"",""ELA""}, {""FCHS_26"",""PLA""}]",2,"""ENSG00000237491.11""","""LINC01409""","""lncRNA""",10,4,"[{""FCHS_19"",""PLA""}, {""FCHS_20"",""ELA""}]",2,0,0,0
"""chr1""",765078,827475,"""ENST00000743835.1""","""-""","""chr1""",778771,810065,"""ENST00000669749.2""","""+""",31294,0.501531,1.0,"""ENSG00000225880.7""","""LINC00115""","""lncRNA""",-6,-1,"[{""FCHS_25"",""ELA""}, {""FCHS_26"",""PLA""}]",2,"""ENSG00000237491.11""","""LINC01409""","""lncRNA""",2,2,"[{""FCHS_19"",""PLA""}, {""FCHS_20"",""ELA""}]",2,0,0,0
"""chr1""",765078,827475,"""ENST00000743835.1""","""-""","""chr1""",778795,810062,"""ENST00000658648.1""","""+""",31267,0.501098,1.0,"""ENSG00000225880.7""","""LINC00115""","""lncRNA""",-6,-1,"[{""FCHS_25"",""ELA""}, {""FCHS_26"",""PLA""}]",2,"""ENSG00000237491.11""","""LINC01409""","""lncRNA""",7,2,"[{""FCHS_19"",""PLA""}, {""FCHS_20"",""ELA""}]",2,0,0,0
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""chrX""",155071502,155123074,"""ENST00000369462.5""","""+""","""chrX""",155065320,155147775,"""ENST00000362018.2""","""-""",51572,1.0,0.625456,"""ENSG00000185515.16""","""BRCC3""","""protein_coding""",-65,-41,"[{""FCHS_446828"",""PLA""}, {""FCHS_593833"",""PLA""}, {""FCHS_446830"",""ELA""}]",3,"""ENSG00000214827.12""","""MTCP1""","""protein_coding""",4,3,"[{""FCHS_446836"",""PLA""}]",1,0,0,0
"""chrX""",155071507,155123077,"""ENST00000330045.12""","""+""","""chrX""",155065162,155147937,"""ENST00000476116.1""","""-""",51570,1.0,0.623014,"""ENSG00000185515.16""","""BRCC3""","""protein_coding""",7,2,"[{""FCHS_446828"",""PLA""}, {""FCHS_593833"",""PLA""}, {""FCHS_446830"",""ELA""}]",3,"""ENSG00000214827.12""","""MTCP1""","""protein_coding_CDS_not_defined""",-9,4,"[{""FCHS_446836"",""PLA""}]",1,0,0,0
"""chrX""",155071507,155123077,"""ENST00000330045.12""","""+""","""chrX""",155065320,155147775,"""ENST00000362018.2""","""-""",51570,1.0,0.625432,"""ENSG00000185515.16""","""BRCC3""","""protein_coding""",7,2,"[{""FCHS_446828"",""PLA""}, {""FCHS_593833"",""PLA""}, {""FCHS_446830"",""ELA""}]",3,"""ENSG00000214827.12""","""MTCP1""","""protein_coding""",4,3,"[{""FCHS_446836"",""PLA""}]",1,0,0,0


In [11]:
bedfile_intersect_antisense_join.write_parquet(
    "../../out/transcript_classification/antisense_transcript/antisense_transcript_join_HNscore_fantabio_v261010.parquet",
    compression="zstd",
    compression_level=9
)